# S-pre

Response diagrams, target quantiles and class labels.


In [ ]:
import os, glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import savgol_filter

# structure model IDs; replace with your own
TRAIN_SET = []        # e.g. ['S1', 'S2', 'S3']
VALID_SET = []        # e.g. ['S4']

RAW_DIR    = 'data/gm/'
RDTD_DIR   = 'data/rdtd/'
TARGET_DIR = 'data/target/'

CFG = dict(f_min=0.1, f_max=10.0, n_freq=490, dt=0.01, duration=40.0,
           zeta=0.05, sg_win=51, sg_poly=3, levels=50, dpi=300, size=(8, 3))

for d in (RDTD_DIR, TARGET_DIR):
    os.makedirs(d, exist_ok=True)


In [ ]:
def sdof(omega, dt, acc, zeta=None):
    """Displacement of a damped SDOF oscillator under base acceleration.

    Consistent central-difference discretisation: the recursion divides by
    ``1 + zeta*omega*dt`` AND damps the ``u[j-1]`` term by ``1 -
    zeta*omega*dt``. The previous version omitted the second factor, which
    compounds a per-step artificial decay and biased the peak response by tens
    of percent (measured: -80% at 1 Hz, -29% at 2 Hz with zeta=0.05).
    """
    zeta = CFG['zeta'] if zeta is None else zeta
    u = np.zeros_like(acc)
    for j in range(1, len(acc) - 1):
        u[j + 1] = (2 * u[j] - (1 - zeta * omega * dt) * u[j - 1]
                    - dt**2 * (acc[j] + omega**2 * u[j])) / (1 + zeta * omega * dt)
    return u


def sdof_batch(omegas, dt, acc, zeta=CFG['zeta']):
    """SDOF displacement for all frequencies at once.

    Identical recursion to :func:`sdof`, vectorised over frequencies, so the
    ``(n_freq, n_steps)`` diagram is built in one pass - orders of magnitude
    fewer Python iterations than the per-frequency loop.
    """
    omegas = np.asarray(omegas, float)
    damp = zeta * omegas * dt
    d = 1.0 + damp
    k = 1.0 - damp
    w2 = omegas**2

    us = np.zeros((len(acc), omegas.size))   # u[j + 1] written at index j + 1
    uprev = np.zeros(omegas.size)
    ucur = np.zeros(omegas.size)
    for j in range(1, len(acc) - 1):
        us[j + 1] = (2 * ucur - k * uprev - dt**2 * (acc[j] + w2 * ucur)) / d
        uprev, ucur = ucur, us[j + 1]
    return us                                # (n_steps, n_freq)


def response_diagram(path, cfg=CFG):
    """Elastic response diagram in the time domain.

    The peak ground acceleration is aligned to the horizontal centre of a
    ``duration`` window; the vertical axis spans ``f_min``-``f_max`` Hz on a
    logarithmic scale, and each cell holds the SDOF displacement normalised by
    its peak.
    """
    acc = np.loadtxt(path, skiprows=1)
    if acc.ndim > 1:                        # e.g. time, acceleration -> last column
        acc = acc[:, -1]
    if len(acc) <= cfg['sg_win']:
        raise ValueError(f'{path}: too short for sg_win={cfg["sg_win"]}')
    if cfg['sg_win'] % 2 == 0:
        raise ValueError(f'sg_win must be odd, got {cfg["sg_win"]}')
    acc = savgol_filter(acc, cfg['sg_win'], cfg['sg_poly'])

    half = int(cfg['duration'] / 2 / cfg['dt'])
    peak = int(np.argmax(np.abs(acc)))
    acc = np.pad(acc, (max(0, half - peak), max(0, peak + half - len(acc))), 'constant')
    start = max(0, peak - half)
    acc = acc[start:start + 2 * half]          # always the full 2*half window

    freqs = np.linspace(cfg['f_min'], cfg['f_max'], cfg['n_freq'])
    z = np.abs(sdof_batch(2 * np.pi * freqs, cfg['dt'], acc))[
        :int(cfg['duration'] / cfg['dt'])].T          # (n_freq, n_steps)
    row_max = z.max(axis=1)
    row_max[row_max == 0] = 1.0
    z /= row_max[:, None]
    z -= z.min()
    z /= (z.max() or 1.0)
    return freqs, z


def save_diagram(freqs, z, out, cfg=CFG):
    plt.figure(figsize=cfg['size'], dpi=cfg['dpi'])
    plt.yscale('log')
    plt.ylim(freqs[0], freqs[-1])
    plt.xlim(0, cfg['duration'])
    plt.contourf(np.linspace(0, cfg['duration'], z.shape[1]), freqs,
                 z, levels=cfg['levels'], cmap='gray')
    plt.axis('off')
    plt.savefig(out, bbox_inches='tight', pad_inches=0)
    plt.close()


In [ ]:
from utils.utils import quantile_curve, label_group, TOP_RATIO


def build_targets(midr, levels=(16, 84)):
    """Target response at the exceedance levels."""
    return {p: quantile_curve(midr, p, method='empirical') for p in levels}


def save_targets(groups_midr, out_dir=TARGET_DIR):
    """Persist per-group target quantiles for S-GMS.

    Writes ``<group>_16.txt`` and ``<group>_84.txt`` into ``out_dir`` - the
    naming convention S-GMS:``target_of`` looks up. ``groups_midr`` maps a
    group id to its ``(n_records, n_floors)`` nonlinear response array. The
    previous version only kept targets in memory, so S-GMS never found them.
    """
    for group, midr in groups_midr.items():
        targets = build_targets(np.asarray(midr, float))
        for lvl, curve in targets.items():
            np.savetxt(os.path.join(out_dir, f'{group}_{lvl}.txt'), curve,
                       fmt='%.6f')


def export_labels(struct, midr, target, out_dir='data/rdtd_labeled',
                  src_dir=RDTD_DIR, index_csv=None, top_ratio=None):
    """Copy one group's diagrams into class folders for S-train.

    Row ``k`` of ``midr`` must follow the sorted raw-record order used by
    :func:`batch` (see ``rdtd_index.csv``). Records labelled 0 by
    :func:`label_group` (closest to ``target``, the ``top_ratio`` fraction)
    go to ``class_a``, the rest to ``class_b`` - matching the alphabetical
    folder order the dataloader assigns to labels 0 and 1.
    """
    import shutil
    if index_csv is None:
        index_csv = os.path.join(TARGET_DIR, 'rdtd_index.csv')
    table = pd.read_csv(index_csv)
    tags = table.loc[table['struct'] == struct, 'tag'].reset_index(drop=True)
    if len(tags) != len(midr):
        raise ValueError(f'{struct}: {len(midr)} responses vs {len(tags)} tags')

    labels, _ = label_group(np.asarray(midr, float), target,
                            top_ratio=TOP_RATIO if top_ratio is None else top_ratio)
    counts = {}
    for lab, tag in zip(labels, tags):
        cls = 'class_a' if lab == 0 else 'class_b'
        dst = os.path.join(out_dir, cls)
        os.makedirs(dst, exist_ok=True)
        src = os.path.join(src_dir, tag + '.png')
        if os.path.exists(src):
            shutil.copy(src, dst)
            counts[cls] = counts.get(cls, 0) + 1
    return counts


def batch(structs=None, limit=None):
    rows = []
    files = sorted(glob.glob(os.path.join(RAW_DIR, '*.txt')))
    for struct in (structs or TRAIN_SET):
        for f in (files[:limit] if limit else files):
            tag = f"{struct}_{os.path.splitext(os.path.basename(f))[0]}"  # <group>_<record>
            try:
                freqs, z = response_diagram(f)
                save_diagram(freqs, z, os.path.join(RDTD_DIR, tag + '.png'))
                np.savetxt(os.path.join(RDTD_DIR, tag + '.txt'), z, fmt='%.4f')
                rows.append(dict(struct=struct, record=os.path.basename(f), tag=tag))
            except Exception as e:
                rows.append(dict(struct=struct, record=os.path.basename(f), error=str(e)))
    table = pd.DataFrame(rows)
    table.to_csv(os.path.join(TARGET_DIR, 'rdtd_index.csv'), index=False)
    return table


# workflow for one structure group:
#   midr    = ...                           # (n_records, n_floors), nonlinear runs
#   targets = build_targets(midr)           # {16: curve, 84: curve}
#   save_targets({'S1': midr})              # -> data/target/S1_16.txt, S1_84.txt
#   export_labels('S1', midr, targets[16])  # -> data/rdtd_labeled/class_a|class_b
